# The metric pays you to delete nodes. I collected, and the leaderboard took it back.

There's a term in the scoring function that rewards predicting fewer objects than expected.

I found it, worked out how to collect it, and offline it looked wonderful. Plus 0.013. Won on every film I tested. Clean mechanism, visible in the numbers.

Then I submitted. Minus 0.004.

This is the whole episode. If you're about to spend slots on node-count tuning, read this first and save yourself four of them.


## The term

The adjusted edge Jaccard is roughly:

```
adj = max(0,  J * (1 - 0.1 * (N_pred - N_est) / N_est))
```

`J` is the edge Jaccard. The bracket penalises predicting more nodes than the organisers' estimate.

Now read it again. It's clipped at zero from below. Nothing clips it from above. Predict *fewer* nodes than `N_est` and the difference goes negative, the multiplier goes above 1, and your score goes up for producing less.

You get paid for deleting. And nothing checks whether what you deleted was wrong.


## Collecting it

Short tracks are the cheapest things to throw away, and the output filter already has a minimum length. So I raised it, and scored on four fixed training films with the competition's own validator.

| min track length | validator (4 films) |
|---|---|
| 6 (baseline) | 0.9368 |
| 8 | 0.9396 |
| **10** | **0.9420** |
| 12 | 0.9324 |
| 15 | 0.9269 |
| 20 | 0.9190 |

Plus 0.0052 at length 10. Winning on all four films. And the baseline, 0.9368, sat right next to my leaderboard score of 0.938, so the validator looked trustworthy.

The mechanism was right there in the numbers too. `J` barely moved. Almost the whole gain came through the node-count multiplier. I wasn't tracking better. I was getting paid to output less.


## It kept getting better, which should have bothered me

The two embryo types have very different label density. One has 0.2 to 1.3 percent of its nodes labelled. The other has 13 to 15. So throwing away short tracks is nearly free on the sparse one and expensive on the dense one.

| configuration | validator |
|---|---|
| baseline | 0.9368 |
| uniform min length 10 | 0.9420 |
| plus detection threshold 0.98 | 0.9450 |
| **per-type: 44b6 = 20, 6bba = 10** | **0.9495** |

On my own pipeline the same policy went 0.9367 to 0.9523.

Every step consistent. Every step mechanistic. Every step reproducible. That is what a real finding looks like, and it is also exactly what this one looked like.


## The leaderboard

| submission | validator | public LB |
|---|---|---|
| baseline fork | 0.9368 | **0.938** |
| min length 10 + det 0.98 + per-type | 0.9495 | **0.934** |
| my own pipeline, same policy | 0.9523 | **0.865** |

Offline said plus 0.013. The leaderboard said minus 0.004.

I can't see the scorer, so I won't pretend to know which explanation is right. The multiplier might be clipped at 1 on the real thing. `N_est` might be computed differently. Or the hidden ground truth is dense enough that the tracks I deleted were mostly real, and `J` fell by as much as the bonus gained.

What I can say is narrower and more useful. The training-film validator does not rank configurations the way the hidden test does, at least along this axis.


## Why the offline check couldn't catch it

This part generalises past this competition, so it's the bit worth keeping.

Training label coverage is low, and wildly uneven between the two embryo types. When you delete a track, the validator only notices if that track happened to be labelled. On the sparse type it usually wasn't. So deletion showed up as pure bonus with no cost attached.

A metric term that rewards absence is the hardest kind to validate on sparse labels. The labels can confirm the reward. They cannot confirm the price.

If you're testing anything that reduces output volume, check what fraction of what you removed was even labelled, before you believe the score.


## Split the score, so you can see which half moved

This is the check I should have been running from the start. It takes your predicted node count and the reference estimate, and tells you how much of your score is the Jaccard and how much is the node-count multiplier.

If a change moves the multiplier while `J` sits still, you are being paid for deleting, and you should assume that money is not in the bank until the leaderboard says so.


In [ ]:
import numpy as np


def split_score(J, n_pred, n_est, k=0.1):
    """Break the adjusted edge Jaccard into its two parts.

    adj = max(0, J * (1 - k * (n_pred - n_est) / n_est))

    Returns a dict with the raw Jaccard, the node-count multiplier, and the
    final score, so you can see which one a change actually moved.
    """
    mult = 1.0 - k * (n_pred - n_est) / n_est
    return {"J": J, "multiplier": mult, "adj": max(0.0, J * mult),
            "node_delta_pct": 100.0 * (n_pred - n_est) / n_est}


def compare(before, after):
    """before/after are dicts from split_score. Says where the change came from."""
    dJ = after["J"] - before["J"]
    dM = after["multiplier"] - before["multiplier"]
    total = after["adj"] - before["adj"]
    share = abs(dM) / (abs(dJ) + abs(dM) + 1e-12)
    flag = "  <-- almost all multiplier, treat as unproven" if share > 0.7 else ""
    return (f"adj {total:+.4f}   from J {dJ:+.4f}, from multiplier {dM:+.4f}"
            f"   ({share:.0%} multiplier){flag}")


# my own case: raising min track length from 6 to 10
before = split_score(J=0.9368, n_pred=100_000, n_est=100_000)
after  = split_score(J=0.9366, n_pred= 95_000, n_est=100_000)   # J flat, 5% fewer nodes
print("baseline :", {k: round(v, 4) for k, v in before.items()})
print("variant  :", {k: round(v, 4) for k, v in after.items()})
print()
print(compare(before, after))
print()
print("leaderboard said: 0.938 -> 0.934")


## What I'd do with this

Don't spend slots on node-count tuning. Three of mine say the lever isn't there. If you have a leaderboard point that says otherwise I'd genuinely like to see it, because that would be more interesting than this notebook.

Report `J` and the multiplier separately when you score offline. My gain was almost all multiplier with flat `J`. That pattern should have been the warning. I read it as encouragement.

Probe one change per submission. I bundled three and then spent more slots pulling them apart.

And the general one, which cost me four submissions: an offline ruler agreeing with your leaderboard score at the baseline tells you nothing about whether it *ranks* things correctly. Mine matched at 0.9368 against 0.938, and then pointed the wrong way.
